<a href="https://colab.research.google.com/github/daviminati/Trab_Final_IA/blob/main/Sprint_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:

!pip install import-ipynb


!git clone https://github.com/daviminati/Trab_Final_IA.git
import os
os.chdir("Trab_Final_IA")

import import_ipynb
import Sprint_4

model_gpt = Sprint_4.GPTModel(Sprint_4.GPT_CONFIG_124M)
print("GPTModel importado com sucesso da Sprint 4!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 33.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [import-ipynb]
Cloning into 'Trab_Final_IA'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 86 (delta 32), reused 3 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 93.90 KiB | 3.76 MiB/s, done.
Resolving deltas: 100% (32/32), done.
torch version: 2.14.1+cpu
Média após LayerNorm (próxima de 0): [[ 0.  0. -0.]
 [ 0.  0.  0.]]
Variância após LayerNorm (próxima de 1): [[1. 1. 1.]
 [1. 1. 1.]]
Valores de entrada:  tensor([-2., -1.,  0.,  1.,  2.])
Saída após a GELU: tensor([-0.0454, -0.1588,  0.0000,  0.8412,  1.9546])
Shape de entrada na FFN: torch.Size([2, 3, 768])
Shape de saída da FFN:   torch.Size([2, 3, 768])
Entrada no TransformerBlock: torch.Size([2, 3, 768])
Saída do TransformerBlock:   torch.Size([2, 3, 768])
Formato do tensor de entrada (To

## 5.2 Entregável 2: Avaliação do Modelo e Funções de Perda

Nesta seção, implementamos as funções para calcular a perda de entropia cruzada (*Cross-Entropy Loss*) e a **Perplexidade** para os conjuntos de dados de treinamento e validação, permitindo monitorar o aprendizado do modelo.

In [12]:
# Importações fundamentais
import torch
import torch.nn as nn
from importlib.metadata import version

print("torch version:", version("torch"))

# Função para calcular a perda (loss) de um lote (batch)
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)

    logits = model(input_batch)

    # O PyTorch espera logits no formato (Batch * Seq_Len, Vocab_Size)
    # e targets no formato (Batch * Seq_Len)
    loss = nn.functional.cross_entropy(
        logits.flatten(0, 1),
        target_batch.flatten()
    )
    return loss

# Função para calcular a perda média em todo um DataLoader
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.0
    if len(data_loader) == 0:
        return float("nan")

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    model.eval()
    with torch.no_grad():
        for i, (input_batch, target_batch) in enumerate(data_loader):
            if i >= num_batches:
                break
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()

    return total_loss / num_batches

torch version: 2.14.1+cpu


## 5.3 Entregável 3: Rotina de Treinamento do GPT

Implementamos a rotina principal de treinamento baseada no otimizador **AdamW**. O código itera por épocas e lotes, zerando os gradientes (`optimizer.zero_grad()`), calculando a retropropagação (`loss.backward()`) e atualizando os pesos dos tensores do modelo.

In [13]:
# Função principal de treinamento do modelo GPT
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)

            loss.backward()
            optimizer.step()

            tokens_seen += input_batch.numel()
            global_step += 1

            # Avaliação periódica durante a época
            if global_step % eval_freq == 0:
                train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
                val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)

                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)

                print(f"Epoca {epoch+1} (Passo {global_step:03d}): "
                      f"Treino Loss {train_loss:.4f} | Val Loss {val_loss:.4f}")

    return train_losses, val_losses, track_tokens_seen

## 5.4 Entregável 4: Experimentos de Treinamento e Configuração de Dados

Para testar o treinamento na prática sem precisar de um dataset gigantesco, simulamos os tensores de dados de treino/validação e executamos uma sessão rápida de treinamento com o otimizador AdamW, registrando as perdas e a convergência.

In [14]:
from torch.utils.data import Dataset, DataLoader

# Dataset simulado para demonstração prática da Sprint 5
class DummyDataset(Dataset):
    def __init__(self, input_ids, target_ids):
        self.input_ids = input_ids
        self.target_ids = target_ids

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# Configuração do modelo e dados de teste
GPT_CONFIG_124M = {
    "vocab_size": 50257, "context_length": 256, "emb_dim": 768,
    "n_heads": 12, "n_layers": 12, "drop_rate": 0.1, "qkv_bias": False
}

# Importando a classe GPTModel da Sprint 4 (ou redefinindo caso necessário)
# Certifique-se de que as classes da Sprint 4 (TransformerBlock, GPTModel, etc.) já foram executadas nas células acima.

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instanciando o modelo para treino
model_gpt = GPTModel(GPT_CONFIG_124M)
model_gpt.to(device)

optimizer = torch.optim.AdamW(model_gpt.parameters(), lr=0.0004, weight_decay=0.1)

# Criando tensores fictícios de treino para validar a rotina
dummy_inputs = torch.randint(0, 50257, (16, 128))
dummy_targets = torch.randint(0, 50257, (16, 128))
train_dataset = DummyDataset(dummy_inputs, dummy_targets)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

val_dataset = DummyDataset(dummy_inputs[:4], dummy_targets[:4])
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

print(f"Dispositivo de execução: {device}")
print("Iniciando rotina de testes de treinamento...")

# Executando 2 épocas de teste rápido
train_losses, val_losses, tokens_tracked = train_model_simple(
    model=model_gpt, train_loader=train_loader, val_loader=val_loader,
    optimizer=optimizer, device=device, num_epochs=2, eval_freq=2,
    eval_iter=1, start_context="Hello", tokenizer=None
)

Dispositivo de execução: cpu
Iniciando rotina de testes de treinamento...
Epoca 1 (Passo 000): Treino Loss 10.3925 | Val Loss 10.9796
Epoca 1 (Passo 002): Treino Loss 9.3224 | Val Loss 9.4583
Epoca 2 (Passo 004): Treino Loss 9.1897 | Val Loss 8.6010
Epoca 2 (Passo 006): Treino Loss 6.4541 | Val Loss 6.7387


## 5.5 Entregável 5: Estratégias de Geração de Texto (Temperatura e Top-k)

Melhoramos a função de geração básica da Sprint 4 introduzindo controles estatísticos cruciais para a qualidade do texto gerado:
1. **Top-k Sampling:** Restringe a escolha aos $k$ tokens mais prováveis, zerando a probabilidade dos demais.
2. **Temperature Scaling:** Divide os *logits* por um parâmetro $T$.
   * $T < 1$: Deixa a distribuição mais "picoada" (previsível e determinística).
   * $T > 1$: Aumenta a suavidade entre as probabilidades, gerando respostas mais criativas e diversas.

In [15]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, device=None):
    if device is None:
        device = idx.device

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:].to(device)

        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]

        # Filtragem Top-k
        if top_k is not None:
            top_logits, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float('-inf')).to(device), logits)

        # Aplicação de Temperatura
        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            # Caso temperatura seja 0, recai sobre o greedy decoding (argmax)
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        idx = torch.cat((idx, idx_next.to(idx.device)), dim=1)

    return idx

# Teste da função de geração avançada
torch.manual_seed(123)
token_prompt = torch.tensor([[15496, 11, 314, 716]]) # Exemplo: "Hello, I am"

# Geração com Top-k=25 e Temperatura=1.4 (Criativa)
tokens_gerados = generate(
    model=model_gpt, idx=token_prompt, max_new_tokens=5,
    context_size=GPT_CONFIG_124M["context_length"], temperature=1.4, top_k=25, device=device
)

print("Prompt inicial (IDs):", token_prompt)
print("Sequência gerada com amostragem avançada (IDs):", tokens_gerados)

Prompt inicial (IDs): tensor([[15496,    11,   314,   716]])
Sequência gerada com amostragem avançada (IDs): tensor([[15496,    11,   314,   716, 33029, 23678, 39298, 14670, 17253]])


## 5.6 Entregável 6: Salvamento e Carregamento de Pesos (State Dict)

Demonstramos como persistir o conhecimento adquirido pelo modelo salvando o dicionário de estados (`state_dict`) em um arquivo local (`model_checkpoint.pth`) e recarregando-o em uma nova instância limpa do modelo para verificar a integridade da inferência.

In [16]:
import os

# Caminho do arquivo de checkpoint
checkpoint_path = "gpt_model_sprint5.pth"

# 1. Salvando os pesos do modelo treinado
torch.save(model_gpt.state_dict(), checkpoint_path)
print(f" Pesos do modelo salvos com sucesso em: {checkpoint_path}")

# 2. Instanciando um novo modelo limpo (sem treino prévio)
torch.manual_seed(123)
novo_modelo = GPTModel(GPT_CONFIG_124M)
novo_modelo.to(device)

# 3. Carregando os pesos salvos no novo modelo
novo_modelo.load_state_dict(torch.load(checkpoint_path, map_location=device))
novo_modelo.eval()

print(" Pesos carregados com sucesso na nova instância do modelo!")

# Verificando se a inferência funciona perfeitamente com o modelo recarregado
with torch.no_grad():
    teste_logits = novo_modelo(token_prompt.to(device))
print(f"Dimensão dos logits gerados pelo modelo carregado: {teste_logits.shape}")

 Pesos do modelo salvos com sucesso em: gpt_model_sprint5.pth
 Pesos carregados com sucesso na nova instância do modelo!
Dimensão dos logits gerados pelo modelo carregado: torch.Size([1, 4, 50257])


## 5.7 Entregável 7: Documentação Técnica e Conclusão da Sprint 5

###  Resumo dos Resultados
1. **Função de Perda:** Acompanhamos a descida do gradiente através da entropia cruzada, validando a eficácia do otimizador AdamW.
2. **Decodificação Avançada:** A combinação de `temperature` e `top-k` permitiu controlar com precisão a diversidade e a repetição nas respostas do modelo.
3. **Persistência:** O arquivo `.pth` gerado comprova que o estado do modelo pode ser armazenado e restaurado sem perdas estruturais.

---

###  Divisão de Trabalho da Dupla

* **Integrante 1 (Davi Minati):** Implementação do glossário (Entregável 1), rotinas de perda e perplexidade (Entregável 2), script de treinamento com AdamW (Entregável 3) e execução dos testes empíricos de convergência (Entregável 4).
* **Integrante 2 (Elian Duarte):** Desenvolvimento das estratégias avançadas de amostragem com temperatura e top-k (Entregável 5), rotinas de salvamento e carregamento de checkpoints via `state_dict` (Entregável 6) e redação da documentação técnica e relatórios finais (Entregável 7).